# JPEG AI as a Threat to Deepfake Detection

Main notebook. Dataset: **ProGAN** (CNNDetection test set). Detector: **CNNDetection** (pre-trained ResNet50). Codec: **JPEG AI reference software**.

Structure (guidelines): Imports - Globals - Utils - Data - Network - Train - Evaluation.

# Imports

In [ ]:
import os, sys, random, subprocess
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.models import resnet50
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_fscore_support, roc_curve)

from google.colab import drive
drive.mount('/content/drive')
print('Imports OK | device:', 'cuda' if torch.cuda.is_available() else 'cpu')

# Globals

Variables used throughout the notebook (paths, bitrates, seed).

In [ ]:
device        = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED          = 42
BPP_LIST      = [0.12, 0.25, 0.50, 0.75]      # target bitrates (bits per pixel)
VALID_IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp')

DATASET_PATH = '/content/drive/MyDrive/ProGAN_Dataset'         # originals + compressed (on Drive)
PROGAN_ROOT  = '/content/progan_data/progan'                   # ProGAN source (after download)
JPEGAI_DIR   = '/content/jpeg-ai-reference-software'           # JPEG AI reference software
WEIGHTS_DIR  = '/content/drive/MyDrive/cnndetection_weights'   # detector weights cache (on Drive)

print('Globals OK | DATASET_PATH =', DATASET_PATH)

# Utils

Support functions (standard detection metrics).

In [ ]:
def compute_metrics(labels, scores, thr=0.5):
    """Standard metrics for binary detection. scores = P(fake), label fake=1."""
    labels, scores = np.asarray(labels), np.asarray(scores)
    pred = (scores > thr).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, pred, average='binary', zero_division=0)
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    eer = float(fpr[np.nanargmin(np.abs(fpr - fnr))])
    return dict(
        AUC=roc_auc_score(labels, scores),
        AP=average_precision_score(labels, scores),
        Acc=(pred == labels).mean(),
        BalAcc=0.5 * ((pred[labels == 1] == 1).mean() + (pred[labels == 0] == 0).mean()),
        Prec=pr, Rec=rc, F1=f1, EER=eer,
    )

print('Utils OK')

# Data

Data management: ProGAN download, JPEG AI setup, real/fake split construction and compression at multiple bitrates (reproducible pipeline, Objective 1).

**The cells in this section are only needed to (re)generate the compressed dataset.** If it is already on Drive (`DATASET_PATH`), evaluation only needs Imports/Globals/Network/Evaluation.

In [ ]:
# (Data) Download the CNNDetection ProGAN test set from Google Drive
FILE_ID = '1NZufxaZrjapdsJE_jjP0LGfh1piBy_9i'   # progan_testset.zip file

if not os.path.isdir(PROGAN_ROOT):
    !rm -rf /content/progan_data /content/progan_testset.zip
    !{sys.executable} -m gdown {FILE_ID} -O /content/progan_testset.zip
    !mkdir -p /content/progan_data
    !unzip -q /content/progan_testset.zip -d /content/progan_data
!ls /content/progan_data/progan | head

In [ ]:
# (Data) JPEG AI reference software setup (ONLY for compression; ~20-40 min)
if not os.path.exists('/content/miniconda/bin/conda'):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    !bash /content/miniconda.sh -b -p /content/miniconda
os.environ['PATH'] = '/content/miniconda/bin:' + os.environ['PATH']
os.environ.pop('PYTHONPATH', None)
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!apt-get -qq install -y git-lfs doxygen graphviz
!git lfs install
%cd /content
if not os.path.isdir(JPEGAI_DIR):
    !git clone https://gitlab.com/wg1/jpeg-ai/jpeg-ai-reference-software.git
%cd /content/jpeg-ai-reference-software
!git lfs fetch
!git lfs checkout
!make setup_env
!conda env list | grep jpeg_ai_vm || echo 'jpeg_ai_vm env NOT created'
print('JPEG AI setup done.')

In [ ]:
# (Data) Functions: real/fake split construction + JPEG AI compression (encoder with -r)
def build_split_by_marker(root, dataset_path, n_per_class=None, seed=SEED,
                          real_marker='/0_real/', fake_marker='/1_fake/'):
    def gather(marker):
        out = []
        for dp, _, fs in os.walk(root):
            for f in fs:
                if f.lower().endswith(VALID_IMG_EXT):
                    p = os.path.join(dp, f)
                    if marker in p:
                        out.append(p)
        return sorted(out)
    real, fake = gather(real_marker), gather(fake_marker)
    rng = random.Random(seed); rng.shuffle(real); rng.shuffle(fake)
    if n_per_class is not None:
        real, fake = real[:n_per_class], fake[:n_per_class]
    for cls, lst in [('real', real), ('fake', fake)]:
        od = os.path.join(dataset_path, f'{cls}_original'); os.makedirs(od, exist_ok=True)
        n = 0
        for p in lst:
            try:
                Image.open(p).convert('RGB').save(os.path.join(od, f'{cls}_{n:05d}.png')); n += 1
            except Exception as e:
                print('skip', p, e)
        print(f'{cls}_original: {n} (out of {len(lst)})')

def _jpegai_run(cmd, jpegai_dir, env='jpeg_ai_vm'):
    r = subprocess.run(['conda', 'run', '-n', env] + cmd, cwd=jpegai_dir, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('JPEG AI failed: ' + r.stderr[-1500:])

def compress_image_jpegai(src, out_png, bpp, jpegai_dir, profile='base', stream_dir='/content/jpegai_streams'):
    os.makedirs(stream_dir, exist_ok=True)
    cfg = ['cfg/tools_off.json', f'cfg/profiles/{profile}.json']
    stream = os.path.join(stream_dir, os.path.splitext(os.path.basename(src))[0] + f'_{bpp}.bin')
    try:
        _jpegai_run(['python', '-m', 'src.reco.coders.encoder', src, stream, '-r', out_png,
                     '--set_target_bpp', str(int(round(bpp * 100))), '--cfg'] + cfg, jpegai_dir)
    finally:
        if os.path.exists(stream): os.remove(stream)

def compress_dataset_jpegai(dataset_path, bpp_list, jpegai_dir):
    for cls in ('real', 'fake'):
        sd = os.path.join(dataset_path, f'{cls}_original')
        imgs = [f for f in sorted(os.listdir(sd)) if f.lower().endswith(VALID_IMG_EXT)]
        print('===', cls, len(imgs), 'images ===')
        for bpp in bpp_list:
            od = os.path.join(dataset_path, f'{cls}_bpp{bpp}'); os.makedirs(od, exist_ok=True)
            for fn in imgs:
                out = os.path.join(od, os.path.splitext(fn)[0] + '.png')
                if os.path.exists(out):
                    continue
                try:
                    compress_image_jpegai(os.path.join(sd, fn), out, bpp, jpegai_dir)
                except Exception as e:
                    print('   ', fn, e)
    print('Compression done.')

print('Data functions OK')

In [ ]:
# (Data) Generate the compressed ProGAN dataset (AFTER download + JPEG AI setup)
os.environ['PATH'] = '/content/miniconda/bin:' + os.environ['PATH']
build_split_by_marker(PROGAN_ROOT, DATASET_PATH, n_per_class=150)
compress_dataset_jpegai(DATASET_PATH, BPP_LIST, JPEGAI_DIR)

# Network

Pre-trained **CNNDetection** detector (Wang et al. 2020): ResNet50 with a single output (logit -> sigmoid = P(fake)). Two variants: `prob0.1` (little data augmentation, more vulnerable to compression) and `prob0.5` (robust). Weights are cached on Drive.

In [ ]:
# (Network) Download CNNDetection weights (cached on Drive, only the first time)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
urls = {
    'blur_jpg_prob0.1.pth': 'https://www.dropbox.com/s/h7tkpcgiwuftb6g/blur_jpg_prob0.1.pth?dl=1',
    'blur_jpg_prob0.5.pth': 'https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=1',
}
for name, url in urls.items():
    dst = os.path.join(WEIGHTS_DIR, name)
    if not os.path.exists(dst):
        !wget -q '{url}' -O '{dst}'
!ls -la {WEIGHTS_DIR}

In [ ]:
# (Network) Load a detector variant + prediction function
DETECTOR_VARIANT = 'blur_jpg_prob0.1.pth'   # use 'blur_jpg_prob0.5.pth' for the robust detector

cnn = resnet50(num_classes=1)
sd = torch.load(os.path.join(WEIGHTS_DIR, DETECTOR_VARIANT), map_location=device)
sd = sd.get('model', sd)
sd = {(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}
print(cnn.load_state_dict(sd))
cnn = cnn.to(device).eval()

# CNNDetection preprocessing: no resize (preserves the fingerprint), center crop 224, ImageNet normalize
cnn_tf = T.Compose([
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def cnndet_predict(path):
    x = cnn_tf(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
    return float(torch.sigmoid(cnn(x).squeeze()))   # P(fake)

print('Detector ready:', DETECTOR_VARIANT)

# Train

The project uses **pre-trained detectors** for the baseline evaluation: there is **no training** in this phase. This section is reserved for the optional fine-tuning for the **mitigation strategies** (objective 3).

# Evaluation

Detector evaluation on `original` and at each JPEG AI compression level. Standard metrics (AUC, AP, Accuracy, Balanced Acc, F1, EER) as a function of bitrate = **degradation curve** (Objective 1).

In [ ]:
# (Evaluation) Per-condition scores + detailed report for the original
def scores_for_condition(suffix):
    s, y = [], []
    for cls, lab in [('real', 0), ('fake', 1)]:
        d = os.path.join(DATASET_PATH, f'{cls}_{suffix}')
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            s.append(cnndet_predict(os.path.join(d, f))); y.append(lab)
    return np.array(y), np.array(s)

y, s = scores_for_condition('original')
print(f'ProGAN - ORIGINAL  (n={len(y)})')
for k, v in compute_metrics(y, s).items():
    print(f'  {k:7s} {v:.3f}')

In [ ]:
# (Evaluation) Standard metrics as a function of bitrate (table)
cols = ['AUC', 'AP', 'Acc', 'BalAcc', 'F1', 'EER']
print('CONDITION'.ljust(12) + ' ' + ' '.join(c.rjust(7) for c in cols) + '    n')
results = {}
for suf in ['original'] + [f'bpp{b}' for b in BPP_LIST]:
    y, s = scores_for_condition(suf)
    if len(y) == 0:
        print(suf.ljust(12) + '  (missing)'); continue
    m = compute_metrics(y, s); results[suf] = m
    print(suf.ljust(12) + ' ' + ' '.join(f'{m[c]:7.3f}' for c in cols) + f'    {len(y)}')

In [ ]:
# (Evaluation) Degradation curve plot
plt.figure(figsize=(8, 5))
for mname in ['AUC', 'AP', 'Acc', 'F1']:
    plt.plot(BPP_LIST, [results[f'bpp{b}'][mname] for b in BPP_LIST], 'o-', label=mname)
    plt.axhline(results['original'][mname], ls='--', alpha=0.25)
plt.axhline(0.5, color='k', ls=':', label='chance')
plt.xlabel('bitrate (bpp)'); plt.ylabel('metric'); plt.ylim(0.4, 1.02)
plt.title('Degradation curve - CNNDetection vs JPEG AI')
plt.legend(); plt.grid(alpha=0.3); plt.show()